In [0]:
# MAGIC %md
# ============================================================
# NOTEBOOK   : gold_customer_segments
# PURPOSE    : Revenue and customer distribution by state/region
# SOURCE     : silver.orders, silver.order_payments, silver.customers
# DESTINATION: fincompliance_catalog.gold.customer_segments
# METRICS:
#   - Unique customers per state
#   - Total revenue per state
#   - Average order value per state
#   - Customer count per region
# ============================================================

In [0]:
%run ../configs/config

In [0]:
authenticate_spark(spark)
set_catalog(spark)

In [0]:
from pyspark.sql.functions import (
    col,
    count,
    countDistinct,
    sum as spark_sum,
    avg,
    round as spark_round,
    rank,
    current_timestamp
)
from pyspark.sql.window import Window

In [0]:
# ============================================================
# CELL 5 - READ FROM SILVER
# ============================================================

df_orders = spark.table(f"{SILVER_DB}.orders")
df_payments = spark.table(f"{SILVER_DB}.order_payments")
df_customers = spark.table(f"{SILVER_DB}.customers")

print("=" * 60)
print("SILVER TABLES LOADED")
print("=" * 60)
print(f"orders          : {df_orders.count():,} rows")
print(f"order_payments  : {df_payments.count():,} rows")
print(f"customers       : {df_customers.count():,} rows")

print("\ncustomers columns:")
for c in df_customers.columns:
    print(f"  {c}")

In [0]:
# ============================================================
# CALCULATE CUSTOMER SEGMENTS BY STATE
# Join orders with customers and payments
# Filter delivered orders only
# Group by customer state
# ============================================================

df_customer_segments = (
    df_orders
    .filter(col("order_status") == "delivered")
    .join(
        df_customers.select(
            "customer_id",
            "customer_state",
            "customer_state_name",
            "customer_region"
        ),
        on="customer_id",
        how="left"
    )
    .join(
        df_payments.select("order_id", "payment_value"),
        on="order_id",
        how="left"
    )
    .groupBy(
        "customer_state",
        "customer_state_name",
        "customer_region"
    )
    .agg(
        countDistinct("customer_id")
            .alias("unique_customers"),
        count("order_id").alias("total_orders"),
        spark_round(spark_sum("payment_value"), 2)
            .alias("total_revenue"),
        spark_round(avg("payment_value"), 2)
            .alias("avg_order_value")
    )
    .orderBy(col("total_revenue").desc())
)

print("Customer segments by state:")
df_customer_segments.show(30, truncate=False)

print(f"\nTotal states : {df_customer_segments.count()}")

In [0]:
# ============================================================
# ADD STATE RANKING AND REGION SUMMARY
# ============================================================

window_rank = Window.orderBy(col("total_revenue").desc())

df_customer_segments = df_customer_segments \
    .withColumn(
        "revenue_rank",
        rank().over(window_rank)
    )

print("Top 10 states with rank:")
df_customer_segments \
    .select(
        "revenue_rank",
        "customer_state_name",
        "total_revenue",
        "unique_customers",
        "avg_order_value"
    ) \
    .orderBy("revenue_rank") \
    .show(10, truncate=False)

# Show region level summary
print("\nRevenue by region:")
df_customer_segments \
    .groupBy("customer_region") \
    .agg(
        spark_sum("unique_customers").alias("region_customers"),
        spark_round(spark_sum("total_revenue"), 2)
            .alias("region_revenue")
    ) \
    .orderBy(col("region_revenue").desc()) \
    .show(truncate=False)

print("Done - Ranking and region summary added")

In [0]:
# ============================================================
# REGION SUMMARY (RERUN)
# ============================================================

print("Revenue by region:")
df_customer_segments \
    .groupBy("customer_region") \
    .agg(
        spark_sum("unique_customers").alias("region_customers"),
        spark_round(spark_sum("total_revenue"), 2)
            .alias("region_revenue")
    ) \
    .orderBy(col("region_revenue").desc()) \
    .show(truncate=False)

In [0]:
# ============================================================
# ADD GOLD AUDIT COLUMN
# ============================================================

df_customer_segments = df_customer_segments \
    .withColumn("gold_updated_at", current_timestamp())

print("Final columns:")
for col_name in df_customer_segments.columns:
    print(f"  {col_name}")
print(f"Total states : {df_customer_segments.count()}")

In [0]:
# ============================================================
# WRITE TO GOLD
# ============================================================

(
    df_customer_segments.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{GOLD_DB}.customer_segments")
)

print(f"Written to {GOLD_DB}.customer_segments")
print(f"Rows : {df_customer_segments.count()}")

In [0]:
# ============================================================
# VERIFICATION
# ============================================================
print("=" * 60)
print("GOLD.CUSTOMER_SEGMENTS VERIFICATION")
print("=" * 60)

df_gold = spark.table(f"{GOLD_DB}.customer_segments")
print(f"Total states  : {df_gold.count()}")
print(f"Total columns : {len(df_gold.columns)}")

print("\nTop 5 states by revenue:")
df_gold.orderBy("revenue_rank").show(5, truncate=False)

print("\nTotal revenue across all states:")
total_rev = df_gold.agg(
    spark_round(spark_sum("total_revenue"), 2)
).collect()[0][0]
print(f"  R$ {total_rev:,.2f}")

print("\nTotal unique customers:")
total_cust = df_gold.agg(
    spark_sum("unique_customers")
).collect()[0][0]
print(f"  {total_cust:,}")

print("=" * 60)
print("gold.customer_segments verification complete!")
print("=" * 60)